# PASTIS-R: Benchmark Inspection

PASTIS-R is the dense crop-type segmentation benchmark. One raw patch is 128x128 pixels with Sentinel-2, Sentinel-1, per-pixel labels, and a published fold id. The loader exposes it lazily as 64x64 tiles.


## Setup


In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from dataio.get_input import get_input
from dataio.inspection import benchmark_input_contract_table, benchmark_metadata_table

REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

BENCHMARK_ROOT = REPO / "data" / "input" / "benchmarks"
print("repo:", REPO)
print("benchmark root:", BENCHMARK_ROOT)

repo: /Users/akshithchowdary/Developer/Projects/org/abe/robustness
benchmark root: /Users/akshithchowdary/Developer/Projects/org/abe/robustness/data/input/benchmarks


## Load Benchmark


In [2]:

MAX_SAMPLES = 8
LOAD_KWARGS = {}

bench = None
load_error = None
try:
    bench = get_input("pastis", root=BENCHMARK_ROOT, max_samples=MAX_SAMPLES, shuffle=False, **LOAD_KWARGS)
    print(f"loaded {bench.name}: {bench.n_samples} samples")
except Exception as exc:
    load_error = exc
    print(f"benchmark could not be loaded locally: {type(exc).__name__}: {exc}")

loaded pastis: 32 samples


## Input Contract and Available Metadata


In [3]:
if bench is None:
    display(Markdown(f"Benchmark tables are skipped because the local data could not be read: `{type(load_error).__name__}: {load_error}`"))
else:
    display(Markdown("### Model-facing input contract"))
    display(benchmark_input_contract_table(bench))
    display(Markdown("### Other benchmark metadata available"))
    display(benchmark_metadata_table(bench))

### Model-facing input contract

,field,scope,dtype/type,coverage,example / summary,role
0,s2,tile,float32,1/1,"B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12",Sentinel-2 image series
1,s1,tile,float32,1/1,"VV, VH, VV/VH",Sentinel-1 ascending image series
2,time,patch,int64,1/1,S2/S1 calendar months per native observation,temporal metadata
3,labels,pixel,int64,1/1,semantic crop-type class id,dense supervision


### Other benchmark metadata available

,field,scope,dtype/type,coverage,example / summary,role
0,patch_id,patch,int64,8/8 (100.0%),"10000, 10001, 10002, 10003, 10004, 10005, ...",source patch identifier
1,fold,patch/tile,int64,8/8 (100.0%),"1: 2, 4: 2, 3: 2, 2: 1, 5: 1",spatial split group
2,s2_months,patch,int64 sequence,8/8 (100.0%),"observations per patch: min=43, median=43, max=43",calendar metadata for S2 observations
3,s1_months,patch,int64 sequence,8/8 (100.0%),"observations per patch: min=65, median=65, max=65",calendar metadata for S1 observations
4,tile_size,benchmark,int,1/1,64,dense tiling parameter
5,ignore_index,pixel,int,1/1,19,void label excluded from metrics


## Lazy Tile Inspection


In [4]:
if bench is not None:
    rows = []
    for patch in bench.patches[:8]:
        rows.append({
            "patch_id": patch.patch_id,
            "fold": patch.fold,
            "s2_observations": len(patch.s2_months),
            "s1_observations": len(patch.s1_months),
            "s2_months": list(map(int, patch.s2_months[:8])),
            "s1_months": list(map(int, patch.s1_months[:8])),
        })
    display(pd.DataFrame(rows))

    tile_id, tile_fold, tile, labels = next(bench.iter_tiles())
    print("first tile:", tile_id, "fold", tile_fold)
    print("s2", tile.s2.shape, "s1", tile.s1.shape, "labels", tile.labels.shape)
    print("valid pixels", int(tile.valid.sum()), "/", tile.valid.size)
else:
    print("PASTIS-R tile inspection skipped")

,patch_id,fold,s2_observations,s1_observations,s2_months,s1_months
0,10000,1,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
1,10001,2,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
2,10002,4,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
3,10003,5,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
4,10004,4,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
5,10005,3,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
6,10006,3,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"
7,10007,1,43,65,"[8, 8, 9, 9, 9, 9, 9, 10]","[9, 9, 9, 9, 9, 10, 10, 10]"


first tile: 10000_0_0 fold 1
s2 (43, 10, 64, 64) s1 (65, 3, 64, 64) labels (64, 64)
valid pixels 4020 / 4096


## Takeaways

The repository key/path is `pastis`; the dataset name remains PASTIS-R in prose. Fold ids are the central non-input metadata: random_id uses the published train/val/test assignment, while geographic_ood holds out spatial folds.
